# Try 3: HF Inference API Evaluation
Uses `huggingface_hub.InferenceClient.image_classification` with model `chbh7051/driver-drowsiness-detection` on local parquet test splits.

In [ ]:
import os
from io import BytesIO
from pathlib import Path

from datasets import load_dataset
from huggingface_hub import InferenceClient

# ---------------------------
# CONFIG
# ---------------------------
MODEL_ID = "chbh7051/driver-drowsiness-detection"
ROOTS = [
    "data/raw/n7i5x9__driver-drowsiness-dataset",
    "data/raw/akahana__Driver-Drowsiness-Dataset",
]
LIMIT_PER_SPLIT = 200  # keep small; API call per image

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing in notebook kernel. Set it and rerun.")

client = InferenceClient(provider="auto", api_key=HF_TOKEN)
print("HF client ready")

# ---------------------------
# HELPERS
# ---------------------------
def normalize_label_name(name: str):
    s = str(name).lower().replace("_", " ").replace("-", " ").strip()
    if "non drowsy" in s or "not drowsy" in s:
        return 0
    if "drowsy" in s or "sleep" in s or "fatigue" in s or "yawn" in s:
        return 1
    if "alert" in s or "awake" in s or "open" in s or "normal" in s:
        return 0
    return None

def discover_data_files(root: Path):
    files = {}
    for p in sorted((root / "data").glob("*.parquet")):
        split = p.name.split("-")[0]
        files.setdefault(split, []).append(str(p))
    return files

def pil_to_png_fileobj(img):
    buf = BytesIO()
    img.convert("RGB").save(buf, format="PNG")
    buf.seek(0)
    buf.name = "sample.png"
    return buf

def infer_binary_from_api(image_fileobj):
    out = client.image_classification(image_fileobj, model=MODEL_ID)
    if not out:
        return None

    best = out[0]
    label = getattr(best, "label", None)
    if label is None and isinstance(best, dict):
        label = best.get("label")

    return normalize_label_name(label)

def print_metrics(name, cm, total):
    correct = int(cm[0][0] + cm[1][1])
    acc = correct / total if total > 0 else 0.0
    tn, fp = cm[0][0], cm[0][1]
    fn, tp = cm[1][0], cm[1][1]

    print(f"\n=== {name} ===")
    print(f"samples={total}")
    print(f"accuracy={acc:.4f}")
    print("confusion_matrix (rows=true, cols=pred) [alert, drowsy]:")
    print([[tn, fp], [fn, tp]])
    print(f"TN={tn} FP={fp} FN={fn} TP={tp}")

# ---------------------------
# EVALUATE BOTH DATASETS (TEST SPLITS)
# ---------------------------
overall_cm = [[0, 0], [0, 0]]
overall_total = 0

for root_str in ROOTS:
    root = Path(root_str)
    data_files = discover_data_files(root)

    if "test" not in data_files:
        print(f"Skipping {root.name}: no test split found")
        continue

    ds = load_dataset("parquet", data_files={"test": data_files["test"]})["test"]
    label_names = ds.features["label"].names
    label_map = {i: normalize_label_name(n) for i, n in enumerate(label_names)}

    cm = [[0, 0], [0, 0]]
    total = 0

    n = len(ds)
    if LIMIT_PER_SPLIT > 0:
        n = min(n, LIMIT_PER_SPLIT)

    for i in range(n):
        row = ds[i]
        y_true = label_map.get(int(row["label"]))
        if y_true is None:
            continue

        img_file = pil_to_png_fileobj(row["image"])
        y_pred = infer_binary_from_api(img_file)
        if y_pred not in (0, 1):
            continue

        cm[y_true][y_pred] += 1
        total += 1

    print_metrics(root.name, cm, total)

    overall_cm[0][0] += cm[0][0]
    overall_cm[0][1] += cm[0][1]
    overall_cm[1][0] += cm[1][0]
    overall_cm[1][1] += cm[1][1]
    overall_total += total

print_metrics("OVERALL", overall_cm, overall_total)
